In [ ]:
!unzip -q "/content/drive/MyDrive/Data/Archive.zip" -d "/content/dataset"


replace /content/dataset/kneeKL299/val/4/9235666R.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset/kneeKL299/val/4/9115049L.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset/kneeKL299/val/4/9177337R.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import torch
import torch.nn as nn

class RelationalTokenCoupling(nn.Module):
    def __init__(self, in_channels=768, token_dim=256, num_heads=4, dropout=0.1, use_global_context=True):
        super(RelationalTokenCoupling, self).__init__()
        self.use_global_context = use_global_context
        self.token_dim = token_dim

        # Project features into shared token space
        self.proj = nn.Linear(in_channels, token_dim)

        # Learnable Role/Positional Embeddings
        self.medial_role_emb = nn.Parameter(torch.randn(1, 1, token_dim))
        self.lateral_role_emb = nn.Parameter(torch.randn(1, 1, token_dim))
        if self.use_global_context:
            self.global_role_emb = nn.Parameter(torch.randn(1, 1, token_dim))

        # Multi-head Self-Attention
        self.mha = nn.MultiheadAttention(embed_dim=token_dim, num_heads=num_heads, dropout=dropout, batch_first=True)

        # Output projection (attended medial + attended lateral -> coupled embedding)
        self.out_proj = nn.Linear(token_dim * 2, token_dim)

        self.norm = nn.LayerNorm(token_dim)
        self.dropout = nn.Dropout(dropout)

        # Store weights for ablation inspection
        self.last_attn_weights = None

    def forward(self, medial_feat, lateral_feat, global_feat=None):
        # Project to token space and add role embeddings
        m_token = self.proj(medial_feat).unsqueeze(1) + self.medial_role_emb
        l_token = self.proj(lateral_feat).unsqueeze(1) + self.lateral_role_emb

        if self.use_global_context and global_feat is not None:
            g_token = self.proj(global_feat).unsqueeze(1) + self.global_role_emb
            tokens = torch.cat([m_token, l_token, g_token], dim=1)
        else:
            tokens = torch.cat([m_token, l_token], dim=1)

        # Self-Attention
        attn_output, attn_weights = self.mha(tokens, tokens, tokens)
        self.last_attn_weights = attn_weights

        attn_output = self.norm(attn_output + tokens)

        # Extract attended tokens and couple them
        m_attended = attn_output[:, 0, :]
        l_attended = attn_output[:, 1, :]

        combined = torch.cat([m_attended, l_attended], dim=-1)
        coupled_embedding = self.out_proj(combined)

        return self.dropout(coupled_embedding)

    @property
    def attn_weights(self):
        return self.last_attn_weights

In [ ]:
# Test Block with Positional Encodings and Attention Inspection
def test_rtc():
    batch_size = 4
    in_channels = 768

    rtc_module = RelationalTokenCoupling(in_channels=in_channels, token_dim=256, use_global_context=True)

    m_feat = torch.randn(batch_size, in_channels)
    l_feat = torch.randn(batch_size, in_channels)
    g_feat = torch.randn(batch_size, in_channels)

    output = rtc_module(m_feat, l_feat, g_feat)
    weights = rtc_module.attn_weights

    print(f"Output coupled embedding shape: {output.shape}")
    print(f"Attention weights shape: {weights.shape}") # Should be (B, 3, 3) with global context

    assert output.shape == (batch_size, 256)
    assert weights.shape == (batch_size, 3, 3)

test_rtc()

Output coupled embedding shape: torch.Size([4, 256])
Attention weights shape: torch.Size([4, 3, 3])


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def validate_rtc_with_backbone():
    print("Starting E7 Validation: RTC with ConvNeXt Tiny Backbone...")

    # 1 & 2. Load ConvNeXt Tiny and strip classifier
    backbone = models.convnext_tiny(weights=None)
    backbone.classifier = nn.Identity() # Keeps output from the final LayerNorm after global pool
    backbone.eval()

    # 3. Dummy input batch
    batch_size = 4
    dummy_img = torch.randn(batch_size, 3, 224, 224)

    # 4. Get pooled global feature (B, 768)
    with torch.no_grad():
        global_feat = backbone(dummy_img).flatten(1)

    # 5. Simulate medial/lateral features with small noise
    medial_feat = global_feat + torch.randn_like(global_feat) * 0.01
    lateral_feat = global_feat + torch.randn_like(global_feat) * 0.01

    # 6. Initialize RTC
    rtc = RelationalTokenCoupling(in_channels=768, token_dim=256, num_heads=4, use_global_context=True)
    rtc.eval() # Set RTC to evaluation mode to disable dropout for attention weights check

    # Forward Pass
    output = rtc(medial_feat, lateral_feat, global_feat)
    weights = rtc.attn_weights

    # 7. Assert output shape
    assert output.shape == (batch_size, 256), f"Expected (4, 256), got {output.shape}"

    # 8. Assert attention weights integrity
    assert weights is not None, "Attention weights were not captured."
    # Softmax check: weights across the key dimension (last axis) should sum to 1
    sum_check = torch.allclose(weights.sum(dim=-1), torch.ones(batch_size, 3), atol=1e-5)
    assert sum_check, "Attention weights do not sum to 1 across the token axis."

    # 9. Print Success
    print(f"E7 RTC PASS")
    print(f"Coupled Embedding Shape: {output.shape}")
    print(f"Attention Weights Shape: {weights.shape}")

if __name__ == '__main__':
    validate_rtc_with_backbone()

Starting E7 Validation: RTC with ConvNeXt Tiny Backbone...
E7 RTC PASS
Coupled Embedding Shape: torch.Size([4, 256])
Attention Weights Shape: torch.Size([4, 3, 3])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PrimaryKLHead(nn.Module):
    """Predicts the 5-class Kellgren-Lawrence (KL) grade for the whole joint."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 5)

    def forward(self, x):
        return F.log_softmax(self.fc(x), dim=-1)

class CORALOrdinalHead(nn.Module):
    """Produces raw logits for 4 binary classifiers representing KL ordinal levels 1-4."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 4)

    def forward(self, x):
        return self.fc(x)

class MetricEmbeddingHead(nn.Module):
    """Produces an L2-normalized embedding for Supervised Contrastive (SupCon) loss."""
    def __init__(self, in_dim=512, embed_dim=128):
        super().__init__()
        self.fc = nn.Linear(in_dim, embed_dim)

    def forward(self, x):
        feat = self.fc(x)
        return F.normalize(feat, p=2, dim=1)

class MedialJSNHead(nn.Module):
    """Grades Joint Space Narrowing (JSN) in the medial compartment (4 classes)."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 4)

    def forward(self, x):
        return F.log_softmax(self.fc(x), dim=-1)

class LateralJSNHead(nn.Module):
    """Grades Joint Space Narrowing (JSN) in the lateral compartment (4 classes)."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 4)

    def forward(self, x):
        return F.log_softmax(self.fc(x), dim=-1)

class OsteophyteHeads(nn.Module):
    """Collection of 4 heads grading osteophytes (none/mild/severe) per ROI patch."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.heads = nn.ModuleList([nn.Linear(in_dim, 3) for _ in range(4)])

    def forward(self, x):
        # Returns list of 4 tensors, each (B, 3)
        return [F.log_softmax(head(x), dim=-1) for head in self.heads]

class UncertaintyHead(nn.Module):
    """Outputs a non-negative scalar representing the model's predictive uncertainty."""
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)

    def forward(self, x):
        return F.softplus(self.fc(x))

In [ ]:
def test_e8_heads():
    print("E8 Head Validation Output Shapes:")
    in_dim = 512
    batch_size = 4
    dummy_input = torch.randn(batch_size, in_dim)

    h1 = PrimaryKLHead(in_dim)
    h2 = CORALOrdinalHead(in_dim)
    h3 = MetricEmbeddingHead(in_dim, 128)
    h4 = MedialJSNHead(in_dim)
    h5 = LateralJSNHead(in_dim)
    h6 = OsteophyteHeads(in_dim)
    h7 = UncertaintyHead(in_dim)

    print(f"H1 KL Grade: {h1(dummy_input).shape}")
    print(f"H2 CORAL: {h2(dummy_input).shape}")
    print(f"H3 Metric: {h3(dummy_input).shape}")
    print(f"H4 Medial JSN: {h4(dummy_input).shape}")
    print(f"H5 Lateral JSN: {h5(dummy_input).shape}")

    h6_out = h6(dummy_input)
    print(f"H6 Osteophytes: {[out.shape for out in h6_out]}")

    print(f"H7 Uncertainty: {h7(dummy_input).shape}")

test_e8_heads()

E8 Head Validation Output Shapes:
H1 KL Grade: torch.Size([4, 5])
H2 CORAL: torch.Size([4, 4])
H3 Metric: torch.Size([4, 128])
H4 Medial JSN: torch.Size([4, 4])
H5 Lateral JSN: torch.Size([4, 4])
H6 Osteophytes: [torch.Size([4, 3]), torch.Size([4, 3]), torch.Size([4, 3]), torch.Size([4, 3])]
H7 Uncertainty: torch.Size([4, 1])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiTaskLoss(nn.Module):
    def __init__(self, config=None, active_heads=None):
        super(MultiTaskLoss, self).__init__()

        # 1. & 2. Load weights from config or fall back to defaults
        default_weights = {
            'h1': 1.0, 'h2': 0.5, 'h3': 0.3,
            'h4': 0.4, 'h5': 0.4, 'h6': 0.3, 'h7': 0.2
        }

        # Map head names to the internal weight keys for consistency
        config_weights = config.get('loss', {}).get('weights', {}) if config else {}
        self.weights = {k: config_weights.get(k, default_weights[k]) for k in default_weights}

        # 4. Toggle mechanism
        self.active_heads = active_heads if active_heads is not None else list(self.weights.keys())

        self.ce_kl = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.ce_jsn = nn.CrossEntropyLoss(ignore_index=-1)
        self.bce_coral = nn.BCEWithLogitsLoss()
        self.mse = nn.MSELoss()

    def summarize(self):
        """3. Prints a table of head configurations."""
        print(f"{'Head':<15} | {'Weight':<10} | {'Active':<8}")
        print("-" * 38)
        for head, weight in self.weights.items():
            is_active = head in self.active_heads and weight > 0
            print(f"{head:<15} | {weight:<10.2f} | {str(is_active):<8}")

    def compute_coral_loss(self, logits, targets):
        batch_size = targets.size(0)
        ordinal_target = torch.zeros(batch_size, 4).to(targets.device)
        for i in range(batch_size):
            if targets[i] > 0:
                ordinal_target[i, :targets[i].long()] = 1
        return self.bce_coral(logits, ordinal_target)

    def compute_supcon_proxy(self, embeddings, labels):
        sim_matrix = F.cosine_similarity(embeddings.unsqueeze(1), embeddings.unsqueeze(0), dim=2)
        mask = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
        mask.fill_diagonal_(0)
        if mask.sum() == 0: return torch.tensor(0.0, device=embeddings.device)
        pos_sim = (sim_matrix * mask).sum() / mask.sum()
        return (1.0 - pos_sim) * 0.5

    def forward(self, preds, labels):
        loss_dict = {}
        total_loss = torch.tensor(0.0, device=labels['kl'].device)

        # Weighted calculation for active heads only
        if 'h1' in self.active_heads:
            loss_dict['kl'] = self.ce_kl(preds['h1'], labels['kl'])
            total_loss += self.weights['h1'] * loss_dict['kl']

        if 'h2' in self.active_heads:
            loss_dict['coral'] = self.compute_coral_loss(preds['h2'], labels['kl'])
            total_loss += self.weights['h2'] * loss_dict['coral']

        if 'h3' in self.active_heads:
            loss_dict['supcon'] = self.compute_supcon_proxy(preds['h3'], labels['kl'])
            total_loss += self.weights['h3'] * loss_dict['supcon']

        if 'h4' in self.active_heads:
            loss_dict['jsn_med'] = self.ce_jsn(preds['h4'], labels['jsn_med'])
            total_loss += self.weights['h4'] * loss_dict['jsn_med']

        if 'h5' in self.active_heads:
            loss_dict['jsn_lat'] = self.ce_jsn(preds['h5'], labels['jsn_lat'])
            total_loss += self.weights['h5'] * loss_dict['jsn_lat']

        if 'h6' in self.active_heads:
            loss_dict['osteophyte'] = torch.stack([self.ce_jsn(h, labels['osteophyte'][:, i])
                                                 for i, h in enumerate(preds['h6'])]).mean()
            total_loss += self.weights['h6'] * loss_dict['osteophyte']

        if 'h7' in self.active_heads:
            probs = F.softmax(preds['h1'], dim=-1)
            entropy = -(probs * torch.log(probs + 1e-6)).sum(dim=-1, keepdim=True)
            loss_dict['uncertainty'] = self.mse(preds['h7'], entropy.detach())
            total_loss += self.weights['h7'] * loss_dict['uncertainty']

        loss_dict['total'] = total_loss
        return loss_dict

In [ ]:
def test_loss_interface():
    # Mock config that might come from your config.yaml
    mock_config = {
        'loss': {
            'weights': {'h1': 1.2, 'h2': 0.6} # Overriding defaults
        }
    }

    print("--- Test 1: All Heads Active (from config) ---")
    criterion_all = MultiTaskLoss(config=mock_config)
    criterion_all.summarize()

    print("\n--- Test 2: Selective Heads (h1, h2 only) ---")
    criterion_subset = MultiTaskLoss(config=mock_config, active_heads=['h1', 'h2'])
    criterion_subset.summarize()

test_loss_interface()

--- Test 1: All Heads Active (from config) ---
Head            | Weight     | Active  
--------------------------------------
h1              | 1.20       | True    
h2              | 0.60       | True    
h3              | 0.30       | True    
h4              | 0.40       | True    
h5              | 0.40       | True    
h6              | 0.30       | True    
h7              | 0.20       | True    

--- Test 2: Selective Heads (h1, h2 only) ---
Head            | Weight     | Active  
--------------------------------------
h1              | 1.20       | True    
h2              | 0.60       | True    
h3              | 0.30       | False   
h4              | 0.40       | False   
h5              | 0.40       | False   
h6              | 0.30       | False   
h7              | 0.20       | False   


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def run_e8_integration_test():
    print("Starting E8 Integration Test...")
    batch_size = 4
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Setup Backbone and Projection
    backbone = models.convnext_tiny(weights=None).to(device)
    backbone.classifier = nn.Identity()
    projector = nn.Linear(768, 512).to(device)

    # 2. Instantiate all 7 heads
    heads = nn.ModuleDict({
        'h1': PrimaryKLHead(in_dim=512),
        'h2': CORALOrdinalHead(in_dim=512),
        'h3': MetricEmbeddingHead(in_dim=512),
        'h4': MedialJSNHead(in_dim=512),
        'h5': LateralJSNHead(in_dim=512),
        'h6': OsteophyteHeads(in_dim=512),
        'h7': UncertaintyHead(in_dim=512)
    }).to(device)

    # 3. Instantiate MultiTaskLoss
    criterion = MultiTaskLoss().to(device)

    # 4. Create Synthetic Labels
    labels = {
        'kl': torch.tensor([0, 1, 2, 3], device=device),
        'jsn_med': torch.tensor([0, 1, -1, 2], device=device),
        'jsn_lat': torch.tensor([1, -1, 0, 3], device=device),
        'osteophyte': torch.randint(-1, 3, (batch_size, 4), device=device)
    }

    # 5. Forward Pass
    dummy_imgs = torch.randn(batch_size, 3, 224, 224, device=device)
    features_768 = backbone(dummy_imgs).flatten(1) # Flatten the backbone output
    fused_features = projector(features_768)

    preds = {}
    for name, head in heads.items():
        preds[name] = head(fused_features)

    # 6. Compute Loss
    loss_results = criterion(preds, labels)
    total_loss = loss_results['total']

    # 7. Backward Pass (Differentiability check)
    total_loss.backward()

    # 8. Assertions
    expected_shapes = {
        'h1': (batch_size, 5),
        'h2': (batch_size, 4),
        'h3': (batch_size, 128),
        'h4': (batch_size, 4),
        'h5': (batch_size, 4),
        'h7': (batch_size, 1)
    }

    for k, shape in expected_shapes.items():
        assert preds[k].shape == shape, f"Shape mismatch for {k}: {preds[k].shape}"
    assert len(preds['h6']) == 4 and preds['h6'][0].shape == (batch_size, 3), "H6 shape mismatch"

    assert not torch.isnan(total_loss), "Total loss is NaN"
    for k, v in loss_results.items():
        assert not torch.isnan(v), f"Loss component {k} is NaN"

    # 9. Print Results
    print("E8 INTEGRATION PASS")
    print(f"Fused Feature Shape: {fused_features.shape}")
    print("Loss components:")
    for k, v in loss_results.items():
        print(f"  - {k}: {v.item():.4f}")

if __name__ == '__main__':
    run_e8_integration_test()

Starting E8 Integration Test...
E8 INTEGRATION PASS
Fused Feature Shape: torch.Size([4, 512])
Loss components:
  - kl: 1.5670
  - coral: 0.6967
  - supcon: 0.0000
  - jsn_med: 1.3688
  - jsn_lat: 1.4000
  - osteophyte: 1.1063
  - uncertainty: 0.8200
  - total: 3.5188


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def run_combined_e7_e8_test():
    print("Starting E7+E8 Combined Integration Test...")
    batch_size = 4
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Setup Architecture Components
    # Backbone
    backbone = models.convnext_tiny(weights=None).to(device)
    backbone.classifier = nn.Identity()

    # E7: RTC
    rtc = RelationalTokenCoupling(in_channels=768, token_dim=256).to(device)

    # Fusion Projection
    # global (768) + rtc (256) = 1024
    projector = nn.Linear(1024, 512).to(device)

    # E8: Heads
    heads = nn.ModuleDict({
        'h1': PrimaryKLHead(in_dim=512),
        'h2': CORALOrdinalHead(in_dim=512),
        'h3': MetricEmbeddingHead(in_dim=512),
        'h4': MedialJSNHead(in_dim=512),
        'h5': LateralJSNHead(in_dim=512),
        'h6': OsteophyteHeads(in_dim=512),
        'h7': UncertaintyHead(in_dim=512)
    }).to(device)

    # Loss
    criterion = MultiTaskLoss().to(device)

    # 2. Simulate 3-Crop Encoding
    global_crop = torch.randn(batch_size, 3, 224, 224, device=device)
    medial_crop = torch.randn(batch_size, 3, 224, 224, device=device)
    lateral_crop = torch.randn(batch_size, 3, 224, 224, device=device)

    g_feat = backbone(global_crop).flatten(1)   # (B, 768)
    m_feat = backbone(medial_crop).flatten(1)   # (B, 768)
    l_feat = backbone(lateral_crop).flatten(1)   # (B, 768)
    print(f"Backbone encoding stage passed. Feat shape: {g_feat.shape}")

    # 3. E7: Relational Token Coupling
    rtc_output = rtc(m_feat, l_feat, g_feat) # (B, 256)
    print(f"E7 RTC stage passed. RTC shape: {rtc_output.shape}")

    # 4. Fusion and Projection
    fused = torch.cat([g_feat, rtc_output], dim=1) # (B, 1024)
    fused_dim_512 = projector(fused)                # (B, 512)
    print(f"Fusion stage passed. Fused shape: {fused_dim_512.shape}")

    # 5. E8: Heads Inference
    preds = {name: head(fused_dim_512) for name, head in heads.items()}

    # 6. Loss and Backward
    labels = {
        'kl': torch.tensor([0, 1, 2, 3], device=device),
        'jsn_med': torch.tensor([0, 1, -1, 2], device=device),
        'jsn_lat': torch.tensor([1, -1, 0, 3], device=device),
        'osteophyte': torch.randint(-1, 3, (batch_size, 4), device=device)
    }

    loss_results = criterion(preds, labels)
    loss_results['total'].backward()

    # 7. Verify Gradient Flow
    first_layer_grad = backbone.features[0][0].weight.grad
    grad_exists = first_layer_grad is not None

    # 8. Report
    print("\n--- E7+E8 COMBINED PASS ---")
    print(f"Global/Medial/Lateral Feats: {g_feat.shape}")
    print(f"RTC Embedding: {rtc_output.shape}")
    print(f"Final Fused Input to Heads: {fused_dim_512.shape}")
    print("Loss Dict:")
    for k, v in loss_results.items():
        print(f"  {k}: {v.item():.4f}")
    print(f"Gradient flow to backbone first layer: {grad_exists}")

    assert grad_exists, "Gradients did not reach the backbone!"

if __name__ == '__main__':
    run_combined_e7_e8_test()

Starting E7+E8 Combined Integration Test...
Backbone encoding stage passed. Feat shape: torch.Size([4, 768])
E7 RTC stage passed. RTC shape: torch.Size([4, 256])
Fusion stage passed. Fused shape: torch.Size([4, 512])

--- E7+E8 COMBINED PASS ---
Global/Medial/Lateral Feats: torch.Size([4, 768])
RTC Embedding: torch.Size([4, 256])
Final Fused Input to Heads: torch.Size([4, 512])
Loss Dict:
  kl: 1.6100
  coral: 0.7064
  supcon: 0.0000
  jsn_med: 1.3299
  jsn_lat: 1.4198
  osteophyte: 1.1345
  uncertainty: 0.8446
  total: 3.5723
Gradient flow to backbone first layer: True
